# Grad-EM training and fixed-batch dynamics

CPU-only analysis of existing TensorBoard scalars and fixed-batch extractor outputs. This notebook never imports model code, runs a forward/backward pass, or writes experiment artifacts. Expensive extraction remains the responsibility of `scripts/extract_grad_em_diagnostics.py`.

## 1. Setup and run registry

In [ ]:
import gc
import html
import json
import math
import os
import sys
from pathlib import Path

import torch  # CPU deserialization of trusted extractor outputs only

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / 'analysis' / 'grad_em_metrics.py').is_file():
        REPO_ROOT = candidate.resolve()
        break
else:
    raise RuntimeError('Run this notebook from the repository root or analysis/')
sys.path.insert(0, str(REPO_ROOT))

from analysis.grad_em_metrics import summarize_layer
from analysis.tb_utils import align_scalar_series, list_scalar_tags, load_scalar_tag

try:
    import pandas as pd
except ImportError:
    pd = None
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None

def table(records):
    return pd.DataFrame(records) if pd is not None else records


In [ ]:
STORAGE_ROOT = Path(os.environ.get('STOCKYARD', Path.home())).expanduser()
TB_ROOT = Path(os.environ.get(
    'GRAD_EM_TB_ROOT',
    STORAGE_ROOT / 'tensorboard/modded-nanogpt-moe/vista',
)).expanduser()
DIAG_ROOT = Path(os.environ.get(
    'GRAD_EM_DIAG_ROOT',
    STORAGE_ROOT / 'analysis/modded-nanogpt-moe',
)).expanduser()
LAYER = 0
BP_COUNTERFACTUAL_ETA = 0.01

# Add a run here only; all later sections consume this registry. Diagnostic
# entries are directory names under DIAG_ROOT and may be absent or split.
RUNS = {
    'bp': {
        'label': 'BP', 'tb_run': 'bp', 'diag': ('bp_control_full',),
        'eta': None, 'router_optimizer': 'muon',
    },
    'ge-eta001-muon': {
        'label': 'GE eta=.001 / Muon', 'tb_run': 'ge-eta001-muon',
        'diag': ('eta001_collapse',), 'eta': 0.001, 'router_optimizer': 'muon',
    },
    'ge-eta003-muon': {
        'label': 'GE eta=.003 / Muon', 'tb_run': 'ge-eta003-muon',
        'diag': ('eta0003_control', 'eta0003_long_collapse'),
        'eta': 0.003, 'router_optimizer': 'muon',
    },
    'ge-eta01-muon': {
        'label': 'GE eta=.01 / Muon', 'tb_run': 'ge-eta01-muon',
        'diag': None, 'eta': 0.01, 'router_optimizer': 'muon',
    },
    'ge-eta03-muon': {
        'label': 'GE eta=.03 / Muon', 'tb_run': 'ge-eta03-muon',
        'diag': None, 'eta': 0.03, 'router_optimizer': 'muon',
    },
    'ge-eta01-adamw': {
        'label': 'GE eta=.01 / AdamW', 'tb_run': 'ge-eta01-adamw',
        'diag': None, 'eta': 0.01, 'router_optimizer': 'adamw',
    },
}

print('TensorBoard root:', TB_ROOT)
print('Diagnostics root:', DIAG_ROOT)
table([{'run': key, **value} for key, value in RUNS.items()])


In [ ]:
def plot_lines(series, title, *, width=760, height=330):
    """Dependency-free inline SVG for {label: scalar-records}."""
    usable = {label: [r for r in records if r.get('value') is not None and math.isfinite(r['value'])]
              for label, records in series.items()}
    usable = {label: records for label, records in usable.items() if records}
    if not usable:
        print(f'{title}: no available values')
        return None
    xs = [r['step'] for records in usable.values() for r in records]
    ys = [r['value'] for records in usable.values() for r in records]
    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
    xmax = xmax if xmax != xmin else xmin + 1
    pad = 0.05 * (ymax - ymin) if ymax != ymin else max(abs(ymin) * 0.05, 1.0)
    ymin, ymax = ymin - pad, ymax + pad
    left, right, top, bottom = 65, 20, 35, 45
    pw, ph = width - left - right, height - top - bottom
    sx = lambda x: left + pw * (x - xmin) / (xmax - xmin)
    sy = lambda y: top + ph * (ymax - y) / (ymax - ymin)
    colors = ['#2563eb', '#dc2626', '#059669', '#7c3aed', '#ea580c', '#0891b2']
    parts = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}'>",
             "<rect width='100%' height='100%' fill='white'/>",
             f"<text x='{left}' y='20' font-family='sans-serif' font-size='15'>{html.escape(title)}</text>",
             f"<line x1='{left}' y1='{top+ph}' x2='{left+pw}' y2='{top+ph}' stroke='#444'/>",
             f"<line x1='{left}' y1='{top}' x2='{left}' y2='{top+ph}' stroke='#444'/>"]
    for index, (label, records) in enumerate(usable.items()):
        color = colors[index % len(colors)]
        points = ' '.join(f"{sx(r['step']):.2f},{sy(r['value']):.2f}" for r in records)
        parts.append(f"<polyline fill='none' stroke='{color}' stroke-width='1.6' points='{points}'/>")
        parts.append(f"<text x='{left + 150*(index % 4)}' y='{height-8-14*(index//4)}' fill='{color}' font-family='sans-serif' font-size='11'>{html.escape(label)}</text>")
    parts.extend([
        f"<text x='{left}' y='{height-25}' font-family='sans-serif' font-size='10'>{xmin}</text>",
        f"<text x='{left+pw-25}' y='{height-25}' font-family='sans-serif' font-size='10'>{xmax}</text>",
        f"<text x='2' y='{top+8}' font-family='sans-serif' font-size='10'>{ymax:.4g}</text>",
        f"<text x='2' y='{top+ph}' font-family='sans-serif' font-size='10'>{ymin:.4g}</text>",
        '</svg>',
    ])
    svg = ''.join(parts)
    if display is not None:
        display(HTML(svg))
    return svg


## 2. Available TensorBoard tags

Tag names below are discovered from each event stream. Missing runs are reported rather than aborting the notebook.

In [ ]:
tag_rows = []
for run_key, spec in RUNS.items():
    path = TB_ROOT / spec['tb_run']
    try:
        tags = list_scalar_tags(path)
    except (FileNotFoundError, ValueError) as error:
        print(f'[skip TensorBoard] {run_key}: {error}')
        continue
    tag_rows.extend({'run': run_key, 'tag': tag} for tag in tags)
table(tag_rows)


## 3. TensorBoard training trajectories

These explicit mappings match the discovered production tags. Sparse tags retain their own steps; no interpolation is performed.

In [ ]:
TB_TAGS = {
    'train_loss': 'metric/loss/train',
    'val_loss': 'metric/loss/val',
    'router_update_rms': 'opt/router/l{layer:02d}/update_rms',
    'router_grad_rms': 'opt/router/l{layer:02d}/grad_rms',
    'router_entropy_norm': 'router/l{layer:02d}/entropy_norm',
    'router_load_cv': 'router/l{layer:02d}/load/cv',
}

def load_tb_metric(metric, runs=RUNS, layer=LAYER):
    tag = TB_TAGS[metric].format(layer=layer)
    loaded = {}
    for run_key, spec in runs.items():
        try:
            records = load_scalar_tag(TB_ROOT / spec['tb_run'], tag)
        except (FileNotFoundError, ValueError) as error:
            print(f'[skip TensorBoard] {run_key}: {error}')
            continue
        if records:
            loaded[spec['label']] = records
        else:
            print(f'[missing tag] {run_key}: {tag}')
    return loaded

for metric in ('train_loss', 'val_loss', 'router_update_rms', 'router_grad_rms',
               'router_entropy_norm', 'router_load_cv'):
    plot_lines(load_tb_metric(metric), f'{metric} (layer {LAYER})')


## 4. Fixed-batch diagnostic trajectories

Extractor schema v1 stores, per layer, FP32 `selected_logits`, FP32 task scores `v=<g,h_i>`, int32 `topk_experts`, and FP32 `router_logsumexp`, plus checkpoint step, E/K metadata, and a shared evaluation-batch hash. The summaries below use `a=softmax(selected_logits)` and `q=softmax(selected_logits-eta*v)`. BP uses the explicitly configured counterfactual eta; it does not claim BP was trained with eta. Only the selected layer is summarized, one memory-mapped CPU checkpoint at a time.

In [ ]:
def torch_load_cpu(path):
    kwargs = {'map_location': 'cpu', 'weights_only': False, 'mmap': True}
    try:
        return torch.load(path, **kwargs)
    except TypeError:
        kwargs.pop('mmap')
        return torch.load(path, **kwargs)

def load_diagnostic_run(run_key, spec, layer=LAYER):
    directory_names = spec.get('diag')
    if not directory_names:
        print(f'[missing diagnostics] {run_key}: no directory registered')
        return []
    eta = spec['eta'] if spec['eta'] is not None else BP_COUNTERFACTUAL_ETA
    rows, seen_steps, reference_hash = [], set(), None
    for directory_name in directory_names:
        directory = DIAG_ROOT / directory_name
        manifest_path = directory / 'manifest.json'
        if not manifest_path.is_file():
            print(f'[missing diagnostics] {run_key}: {directory}')
            continue
        manifest = json.loads(manifest_path.read_text())
        if manifest.get('schema_version') != 1:
            raise ValueError(f'unsupported diagnostic schema: {manifest_path}')
        batch_hash = manifest.get('eval_batch_hash')
        reference_hash = reference_hash or batch_hash
        if batch_hash != reference_hash:
            raise ValueError(f'evaluation-batch hash changed within {run_key}')
        for path in sorted(directory.glob('step_*.pt')):
            payload = torch_load_cpu(path)
            step = int(payload['completed_updates'])
            if step in seen_steps:
                raise ValueError(f'duplicate diagnostic step {step} for {run_key}')
            if payload.get('schema_version') != 1 or payload.get('eval_batch_hash') != reference_hash:
                raise ValueError(f'incompatible diagnostic payload: {path}')
            tensors = payload['layers'][layer]
            row = summarize_layer(
                tensors['selected_logits'], tensors['v'], tensors['topk_experts'],
                eta=eta, num_experts=int(payload['model']['num_experts']),
            )
            row.update(run=run_key, label=spec['label'], step=step, layer=layer,
                       router_optimizer=spec['router_optimizer'],
                       counterfactual=(spec['eta'] is None),
                       eval_batch_hash=reference_hash)
            rows.append(row)
            seen_steps.add(step)
            del payload
            gc.collect()
    return sorted(rows, key=lambda row: row['step'])

DIAGNOSTICS = {key: load_diagnostic_run(key, spec) for key, spec in RUNS.items()}
table([row for rows in DIAGNOSTICS.values() for row in rows])


In [ ]:
DIAGNOSTIC_PLOTS = {
    'load_cv': 'fixed-batch load CV',
    'entropy_norm_mean': 'mean H(a) / log(K)',
    'selected_logit_std_mean': 'mean selected-logit std',
    'eta_dv_p99': 'eta * Delta-v p99',
    'eta_dv_p99_9': 'eta * Delta-v p99.9',
    'tv_p99': 'TV(a,q) p99',
    'tv_p99_9': 'TV(a,q) p99.9',
    'r_p99': 'directional amplification R p99',
    'r_p99_9': 'directional amplification R p99.9',
    'tv_gt_0_1_count': 'count(TV > .1)',
    'd_mean': 'mean absolute routing regret D',
    'sigma_v_mean': 'mean local probability-weighted sigma_v',
}

def diagnostic_series(metric):
    return {RUNS[key]['label']: [
        {'step': row['step'], 'value': float(row[metric])}
        for row in rows if metric in row and math.isfinite(float(row[metric]))
    ] for key, rows in DIAGNOSTICS.items() if rows}

for metric, title in DIAGNOSTIC_PLOTS.items():
    plot_lines(diagnostic_series(metric), f'{title} (layer {LAYER})')


## 5. Muon versus AdamW router comparison

The causal comparison is GE eta=.01 with Muon versus AdamW router optimization, holding other settings fixed. Optimizer families normalize updates differently: plots may show both `update_rms` trajectories, but their numerical magnitudes must **not** be interpreted as identical-scale quantities. The AdamW registry entry is intentionally safe before its data exists.

In [ ]:
optimizer_runs = {key: RUNS[key] for key in ('ge-eta01-muon', 'ge-eta01-adamw')}
for metric in ('train_loss', 'val_loss', 'router_update_rms', 'router_grad_rms',
               'router_entropy_norm', 'router_load_cv'):
    plot_lines(load_tb_metric(metric, optimizer_runs),
               f'Muon vs AdamW: {metric} (layer {LAYER})')


## 6. Failure-state analysis

Select a run and inspect its late fixed-batch state without imposing a failure threshold or causal label.

In [ ]:
FAILURE_RUN = 'ge-eta003-muon'
failure_columns = [
    'step', 'load_cv', 'entropy_norm_mean', 'selected_logit_std_mean',
    'eta_dv_p99_9', 'tv_p99_9', 'r_p99_9', 'tv_gt_0_1_count',
    'd_mean', 'sigma_v_mean', 'argmax_a_ne_argmin_v_fraction',
    'min_v_probability_mean', 'current_top_v_gap_mean',
]
failure_rows = DIAGNOSTICS.get(FAILURE_RUN, [])
table([{column: row[column] for column in failure_columns} for row in failure_rows[-12:]])


## 7. Cross-source exact-step alignment

TensorBoard and fixed-batch metrics are joined on exact completed-update values. `inner` is useful for direct comparison; switch to `outer` to expose sparse/missing samples. No interpolation is performed.

In [ ]:
ALIGN_RUN = 'ge-eta003-muon'
ALIGN_HOW = 'inner'

def diagnostic_scalar_records(rows, metric):
    return [{'step': row['step'], 'value': float(row[metric]), 'wall_time': 0.0}
            for row in rows if metric in row and math.isfinite(float(row[metric]))]

spec = RUNS[ALIGN_RUN]
tb_path = TB_ROOT / spec['tb_run']
try:
    update_records = load_scalar_tag(
        tb_path, TB_TAGS['router_update_rms'].format(layer=LAYER))
except (FileNotFoundError, ValueError):
    update_records = []
diag_rows = DIAGNOSTICS.get(ALIGN_RUN, [])
ALIGNED = align_scalar_series({
    'router_update_rms': update_records,
    'eta_dv_p99_9': diagnostic_scalar_records(diag_rows, 'eta_dv_p99_9'),
    'tv_p99_9': diagnostic_scalar_records(diag_rows, 'tv_p99_9'),
    'entropy_norm': diagnostic_scalar_records(diag_rows, 'entropy_norm_mean'),
    'load_cv': diagnostic_scalar_records(diag_rows, 'load_cv'),
}, how=ALIGN_HOW)
table(ALIGNED)
